# Medical ML Model Training
## Disease Prediction Model

In [ ]:
# Import libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, accuracy_score
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

In [ ]:
# Load data
import sys
sys.path.append('../src')
from data.loader import DataLoader

loader = DataLoader()
data = loader.get_sample_medical_data()
print(f"Dataset shape: {data.shape}")

In [ ]:
# Data preprocessing
def preprocess_medical_data(df):
    """Preprocess medical data for ML."""
    df = df.copy()
    
    # Encode categorical variables
    le = LabelEncoder()
    df['gender'] = le.fit_transform(df['gender'])  # M=1, F=0
    
    # One-hot encode cholesterol
    cholesterol_dummies = pd.get_dummies(df['cholesterol'], prefix='chol', drop_first=True)
    df = pd.concat([df, cholesterol_dummies], axis=1)
    
    # Drop original cholesterol column
    df = df.drop('cholesterol', axis=1)
    
    # Drop patient_id (identifier)
    df = df.drop('patient_id', axis=1)
    
    return df

processed_data = preprocess_medical_data(data)
print("Processed data columns:", processed_data.columns.tolist())
processed_data.head()

In [ ]:
# Prepare features and target
X = processed_data.drop('has_disease', axis=1)
y = processed_data['has_disease']

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, 
                                                    test_size=0.2, 
                                                    random_state=42, 
                                                    stratify=y)

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nClass distribution in training:\n{y_train.value_counts(normalize=True)}")

In [ ]:
# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
# Initialize models
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=1000),
    'Random Forest': RandomForestClassifier(random_state=42, n_estimators=100),
    'Gradient Boosting': GradientBoostingClassifier(random_state=42),
    'SVM': SVC(probability=True, random_state=42)
}

In [ ]:
# Train and evaluate models
results = {}

for name, model in models.items():
    # Train model
    model.fit(X_train_scaled, y_train)
    
    # Predictions
    y_pred = model.predict(X_test_scaled)
    y_pred_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    # Calculate metrics
    accuracy = accuracy_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    
    # Store results
    results[name] = {
        'model': model,
        'accuracy': accuracy,
        'roc_auc': roc_auc,
        'predictions': y_pred,
        'probabilities': y_pred_proba
    }
    
    print(f"{name}:")
    print(f"  Accuracy: {accuracy:.4f}")
    print(f"  ROC-AUC: {roc_auc:.4f}")
    print("  " + "-" * 30)

In [ ]:
# Visualize model comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
accuracies = [results[name]['accuracy'] for name in results]
axes[0].bar(results.keys(), accuracies, color=['skyblue', 'lightgreen', 'lightcoral', 'gold'])
axes[0].set_title('Model Accuracy Comparison')
axes[0].set_ylabel('Accuracy')
axes[0].set_ylim([0, 1])
for i, v in enumerate(accuracies):
    axes[0].text(i, v + 0.01, f"{v:.3f}", ha='center')

# ROC-AUC comparison
auc_scores = [results[name]['roc_auc'] for name in results]
axes[1].bar(results.keys(), auc_scores, color=['skyblue', 'lightgreen', 'lightcoral', 'gold'])
axes[1].set_title('ROC-AUC Score Comparison')
axes[1].set_ylabel('ROC-AUC')
axes[1].set_ylim([0, 1])
for i, v in enumerate(auc_scores):
    axes[1].text(i, v + 0.01, f"{v:.3f}", ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# ROC Curves
plt.figure(figsize=(10, 8))

for name in results:
    fpr, tpr, _ = roc_curve(y_test, results[name]['probabilities'])
    plt.plot(fpr, tpr, label=f'{name} (AUC = {results[name]["roc_auc"]:.3f})', linewidth=2)

plt.plot([0, 1], [0, 1], 'k--', label='Random Guess')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curves - Model Comparison')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
# Feature importance for Random Forest
rf_model = results['Random Forest']['model']
feature_importance = pd.DataFrame({
    'feature': X.columns,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 6))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Feature Importance')
plt.title('Random Forest Feature Importance')
plt.gca().invert_yaxis()
plt.tight_layout()
plt.show()

print("Top 5 Important Features:")
print(feature_importance.head())

In [ ]:
# Save best model
import joblib
import os

# Create models directory if not exists
os.makedirs('models', exist_ok=True)

# Get best model based on ROC-AUC
best_model_name = max(results, key=lambda x: results[x]['roc_auc'])
best_model = results[best_model_name]['model']

# Save model and scaler
joblib.dump(best_model, 'models/best_disease_predictor.pkl')
joblib.dump(scaler, 'models/scaler.pkl')

print(f"Saved best model: {best_model_name}")
print(f"Model saved to: models/best_disease_predictor.pkl")